In [ ]:
import os
import tarfile
import numpy as np
import pandas as pd
import tensorflow as tf
from pandas import DataFrame, Series, concat, read_csv
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.models import Sequential, load_model, Model
from keras.metrics import Precision, Accuracy
from keras.layers import LSTM, Dense, Dropout, Input, Attention, GlobalAveragePooling1D
from keras.callbacks import ModelCheckpoint, EarlyStopping, Callback, ReduceLROnPlateau
from keras.optimizers import schedules
from math import sqrt
import matplotlib
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

matplotlib.use('Agg')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import kaleido

print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Rozpakowanie danych (jeśli nie są jeszcze rozpakowane)
import tarfile
if not os.path.exists('./data/2_150x9/2_150x9f.csv'):
    file = tarfile.open('data/2_150x9.tar.gz')
    file.extractall(path='./data/2_150x9/')
    file.close()
    print("Data extracted.")
else:
    print("Data already extracted.")

In [ ]:
def data(time, features):
    # Ekspert: timeStep = 10 (najlepsze okno czasowe z optymalizacji)
    timestepsPerSample = time
    timestepsPerSampleWholeData = 150

    file_features = './data/2_150x9/2_150x9f.csv'
    file_labels = './data/2_150x9/2_150x9l.csv'

    data_strings = np.genfromtxt(file_features, delimiter=';')
    labels_strings = np.genfromtxt(file_labels, delimiter=';')

    # Ekspert: features = 3 (wszystkie znormalizowane cechy z MQL)
    # Używamy wszystkich dostępnych kolumn w pliku
    if features == 3:
        data_s = data_strings[:, :]
    else:
        data_s = data_strings[:, :]
        
    num_features = data_s.shape[1]
    
    X = data_s.astype(float).reshape((-1, timestepsPerSampleWholeData, num_features))
    Y = labels_strings.astype(float).reshape((-1, 6))

    X_mod = X[:, timestepsPerSampleWholeData - timestepsPerSample:]
    Y_mod = Y[:]
    timestepsPerSampleWholeData = X_mod.shape[1]

    # Ekspert: shuffle=False jest krytyczne dla szeregów czasowych
    x_train, x_test, Y_train, Y_test = train_test_split(X_mod, Y_mod, test_size=0.15, shuffle=False)
    y_train = Y_train[:, 0:2]
    y_test = Y_test[:, 0:2]

    print(f"Train shape: {x_train.shape}")
    print(f"Timesteps: {timestepsPerSampleWholeData}")
    print(f"Train Samples: {x_train.shape[0]}")
    print(f"Test Samples: {x_test.shape[0]}")
    print(f"Num features: {num_features}")
    return x_train, x_test, y_train, y_test, Y_test

In [ ]:
def shuffle_weights(model, weights=None):
    if weights is None:
        weights = model.get_weights()
    weights = [np.random.permutation(w.flat).reshape(w.shape) for w in weights]
    model.set_weights(weights)

def fit_lstmModel(i, x_train, y_train, x_test, y_test, batch_size, nb_epoch, neurons, time, dropout, modelVar, learning_rate):
    # Ekspert: modelVar = 5 (Mechanizm Attention)
    if modelVar == 5:
        inputs = Input(shape=(x_train.shape[1], x_train.shape[2]))
        
        # Ekspert: neurons = 150, dropout = 0.2
        lstm_out = LSTM(units=neurons, return_sequences=True)(inputs)
        lstm_out = Dropout(dropout)(lstm_out)
        
        lstm_out2 = LSTM(units=neurons, return_sequences=True)(lstm_out)
        lstm_out2 = Dropout(dropout)(lstm_out2)
        
        # Self-attention
        attention_out = Attention()([lstm_out2, lstm_out2])
        
        # Pool the sequence to a single vector
        pooled = GlobalAveragePooling1D()(attention_out)
        
        dense_out = Dense(units=neurons // 2)(pooled)
        dense_out = Dropout(dropout)(dense_out)
        
        outputs = Dense(y_train.shape[1], activation='sigmoid')(dense_out)
        model = Model(inputs=inputs, outputs=outputs)
    else:
        raise ValueError("This notebook is configured exclusively for modelVar=5 (Attention)")

    # Ekspert: Optymalizator Nadam z learning_rate = 0.001
    optimizer = tf.keras.optimizers.Nadam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    if i == 0:
        model.save_weights("./saved_models/initial.weights.h5")
    elif i > 0:
        shuffle_weights(model, weights=None)
        
    model.summary()

    # Callbacks
    checkpoint = ModelCheckpoint(
        filepath='./saved_models/last_saved_model.keras',
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )

    # Ekspert: EarlyStopping zapobiega przeuczeniu (patience=3)
    earlyStopping = EarlyStopping(
        monitor='val_loss',
        start_from_epoch=5,
        restore_best_weights=True,
        verbose=1,
        patience=3
    )
    
    # Ekspert: ReduceLROnPlateau dla lepszej zbieżności
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=2, 
        min_lr=0.00001, 
        verbose=1
    )

    profitList = list()
    profitListSaveModel = list()
    
    class MyCallback(Callback):
        def on_epoch_end(self, epoch, logs=None):
            predict = model.predict(x_test, batch_size=batch_size, verbose=0)
            profit = funcProfit(predict, Y_test)
            
            # Zapis najlepszego modelu na podstawie PROFITU
            if len(profitListSaveModel) == 0:
                model.save(f'./saved_models/best_model_attention_{str(time)}.keras')
                print(f"New best model saved, previous best profit -> First")
            elif profit > max(profitListSaveModel):
                model.save(f'./saved_models/best_model_attention_{str(time)}.keras')
                print(f"New best model saved, previous best profit -> {str(max(profitListSaveModel))}")
            
            profitList.append(profit)
            profitListSaveModel.append(profit)
            print(f"Profit: {profit}")

    history = model.fit(
        x_train, y_train, 
        epochs=nb_epoch, 
        batch_size=batch_size, 
        shuffle=False, 
        validation_data=(x_test, y_test), 
        callbacks=[checkpoint, earlyStopping, MyCallback(), reduce_lr], 
        verbose=2
    )
    
    profitArray = np.array(profitList)
    return model, history, profitArray

In [ ]:
def funcProfit(predict, Y_test):
    predict_classes = np.where(predict > 0.56, 1, 0)
    concat = np.hstack((predict_classes, Y_test[:, 2:]))
    df = pd.DataFrame(concat, columns=['Sell', 'Buy', 'Close', 'Open', 'High', 'Low'])

    spread = 0.03
    tp = 1000
    sum_profit = 0
    sell = 0
    buy = 0
    
    for i in range(0, len(df)):
        if (i-1) >= 0:
            if df.at[i,'Sell'] > 0.8 and df.at[i-1,'Sell'] < 0.2:
                if buy > 0:
                    sell = df.at[i,'Open'] - spread
                    sum_profit += df.at[i,'Open'] - buy
                    buy = 0
                elif sell == 0:
                    sell = df.at[i,'Open'] - spread
            elif df.at[i,'Sell'] > 0.8 and df.at[i-1,'Sell'] > 0.8 and sell > 0:
                if df.at[i-1,'High'] >= (sell + tp):
                    sum_profit -= tp
                    sell = 0
                if df.at[i-1,'Low'] <= (sell - tp):
                    sum_profit += tp
                    sell = 0

            elif df.at[i,'Buy'] > 0.8 and df.at[i-1,'Buy'] < 0.2:
                if sell > 0:
                    buy = df.at[i,'Open'] + spread
                    sum_profit += sell - df.at[i,'Open']
                    sell = 0
                elif buy == 0:
                    buy = df.at[i,'Open'] + spread
            elif df.at[i,'Buy'] > 0.8 and df.at[i-1,'Buy'] > 0.8 and buy > 0:
                if df.at[i-1,'Low'] <= (buy - tp):
                    sum_profit -= tp
                    buy = 0
                if df.at[i-1,'High'] >= (buy + tp):
                    sum_profit += tp
                    buy = 0

    return round(sum_profit, 2)

def funcProfitTP(predict, Y_test):
    predict_classes = np.where(predict > 0.56, 1, 0)
    concat = np.hstack((predict_classes, Y_test[:, 2:]))
    df = pd.DataFrame(concat, columns=['Sell', 'Buy', 'Close', 'Open', 'High', 'Low'])

    spread = 0.03
    tp = 1000
    sl = 2
    sum_profit = 0
    sell = 0
    buy = 0
    transact = 0

    for i in range(0, len(df)):
        if (i-3) >= 0:
            buyCondition = 0
            sellCondition = 0
            if df.at[i,'Buy'] == 1 and df.at[i-1,'Buy'] == 1 and buy == 0: 
                buyCondition = 1
            if df.at[i,'Sell'] == 1 and df.at[i-1,'Sell'] == 1 and sell == 0: 
                sellCondition = 1

            if df.at[i,'Buy'] == 0:
                if df.at[i-1,'Buy'] == 0:
                    buy = 0
                elif df.at[i-1,'Buy'] == 1 and buy > 0:
                    sum_profit += df.at[i,'Open'] - buy
                    buy = 0 
            elif buyCondition == 1:
                buy = df.at[i,'Open'] + spread
                transact += 1
                            
            if df.at[i,'Sell'] == 0:
                if df.at[i-1,'Sell'] == 0:
                    sell = 0
                elif df.at[i-1,'Sell'] == 1 and sell > 0:
                    sum_profit += sell - df.at[i,'Open']
                    sell = 0
            elif sellCondition == 1:
                sell = df.at[i,'Open'] - spread
                transact += 1

    return round(sum_profit, 2)

In [ ]:
def experiment(repeats, epochs, neurons, time, dropout, modelVar, features, learning_rate, batch_size):
    x_train, x_test, y_train, y_test, Y_test = data(time, features)

    accuracy = list()
    profit = list()
    profitTP = list()
    metrics = list()

    for r in range(repeats):
        print(f"\n--- Repeat {r+1}/{repeats} running ---")
        model, history, profitArray = fit_lstmModel(
            r, x_train, y_train, x_test, y_test, 
            batch_size, epochs, neurons, time, 
            dropout, modelVar, learning_rate
        )
        
        # Wczytanie najlepszego modelu z danego powtórzenia
        model = load_model(filepath=f"./saved_models/best_model_attention_{str(time)}.keras")
        predict = model.predict(x_test, batch_size=batch_size, verbose=0)
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

        metric = pd.DataFrame(history.history)
        metric['epoch'] = history.epoch
        metric = metric.assign(Profit=profitArray)

        metrics.append(metric)
        accuracy.append(test_acc * 100)
        profit.append(funcProfit(predict, Y_test))
        profitTP.append(funcProfitTP(predict, Y_test))
    
    return accuracy, profit, profitTP, metrics

def plotsOut(metrics, title_suffix=""):
    fig = make_subplots(rows=1, cols=3, subplot_titles=('Accuracy', 'Loss', 'Profit'))
    fig.update_layout(autosize=True, width=1600, height=600, title_text=f"Training Metrics {title_suffix}")

    for idx, m in enumerate(metrics):
        fig.add_trace(go.Scatter(x=m['epoch'], y=m['accuracy'], name=f'acc_{idx}', line_color='#0000ff', showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(x=m['epoch'], y=m['val_accuracy'], name=f'val_acc_{idx}', line_color='#EF8260', showlegend=False), row=1, col=1)
        
        fig.add_trace(go.Scatter(x=m['epoch'], y=m['loss'], name=f'loss_{idx}', line_color='#0000ff', showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=m['epoch'], y=m['val_loss'], name=f'val_loss_{idx}', line_color='#EF8260', showlegend=False), row=1, col=2)
        
        fig.add_trace(go.Scatter(x=m['epoch'], y=m['Profit'], name=f'profit_{idx}', line_color='#EF8260', showlegend=False), row=1, col=3)

    fig.update_xaxes(title_text='epochs')
    fig.write_image(file=f"final_attention_model_metrics.jpg", engine="kaleido")
    fig.show()

In [ ]:
# ==========================================
# THE EXECUTION BLOCK - EXPERT CONFIGURATION
# ==========================================

# Ekspert: Parametry globalne zoptymalizowane dla najlepszego zysku
repeats = 3          # Weryfikacja stabilności wyników
nb_epoch = 100       # Zabezpieczone przez EarlyStopping
timeStep = 10        # Najlepsze okno czasowe
features = 3         # Wszystkie 21 znormalizowanych cech z MQL
modelVar = 5         # Mechanizm Attention
batch_size = 64      # Optymalny rozmiar batcha
neurons = 150        # Liczba neuronów w warstwach LSTM
dropout = 0.2        # Zapobieganie przeuczeniu
learning_rate = 0.001 # Początkowy LR dla optymalizatora Nadam

print("Rozpoczynam ostateczne trenowanie modelu Attention...")
accuracy, profit, profitTP, metrics = experiment(
    repeats=repeats, 
    epochs=nb_epoch, 
    neurons=neurons, 
    time=timeStep, 
    dropout=dropout, 
    modelVar=modelVar, 
    features=features, 
    learning_rate=learning_rate,
    batch_size=batch_size
)

print("\n=== PODSUMOWANIE WYNIKÓW ===")
print(f"Accuracy: {accuracy}")
print(f"Profit: {profit}")
print(f"Profit TP: {profitTP}")

plotsOut(metrics, title_suffix="(Attention, TimeStep=10, Features=All)")